In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import fbeta_score, roc_auc_score, make_scorer
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH = '/content/train (2).parquet'
TEST_PATH  = '/content/test (2).parquet'

# ──────────────────────────────────────────
# 1. 전처리
# ──────────────────────────────────────────
df = pd.read_parquet(TRAIN_PATH)
y  = df['Bankrupt?']
X  = df.drop(columns=['Bankrupt?'])

corr_matrix = X.corr().abs()
upper   = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.97)]
X_reduced = X.drop(columns=to_drop)

scaler   = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_reduced), columns=X_reduced.columns)
X_model  = X_scaled.drop(columns=['ID'], errors='ignore')
print(f"전체 피처: {X_model.shape[1]}개 | 파산 비율: {y.mean()*100:.2f}%")

X_train, X_val, y_train, y_val = train_test_split(
    X_model, y, test_size=0.2, random_state=42, stratify=y
)
scale_pos = int((y_train==0).sum() / (y_train==1).sum())
print(f"scale_pos_weight: {scale_pos}")

# ──────────────────────────────────────────
# 2. 도메인 지식 기반 피처 30개
# ──────────────────────────────────────────
must_have = [
    ' ROA(A) before interest and % after tax',
    ' Retained Earnings to Total Assets',
    ' Working Capital to Total Assets',
    ' Working Capital/Equity',
    ' Debt ratio %',
    ' Liability to Equity',
    ' Borrowing dependency',
    ' Interest-bearing debt interest rate',
    ' Current Ratio',
    ' Quick Ratio',
    ' Cash/Current Liability',
    ' Net Income to Stockholder\'s Equity',
    ' Persistent EPS in the Last Four Seasons',
    ' Cash flow rate',
    ' Operating Gross Margin',
]
support = [
    ' Total debt/Total net worth',
    ' Current Liabilities/Equity',
    ' Equity to Long-term Liability',
    ' Net profit before tax/Paid-in capital',
    ' Per Share Net profit before tax (Yuan \xa5)',
    ' Revenue per person',
    ' Operating profit per person',
    ' Cash Flow to Sales',
    ' Total Asset Growth Rate',
    ' Net Value Growth Rate',
    ' After-tax Net Profit Growth Rate',
    ' Accounts Receivable Turnover',
    ' Total Asset Turnover',
    ' CFO to Assets',
    ' Pre-tax net Interest Rate',
]

top_features = [f for f in must_have + support if f in X_model.columns]
missing      = [f for f in must_have + support if f not in X_model.columns]
print(f"선택 피처: {len(top_features)}개")
if missing:
    print(f"없는 컬럼: {[f.strip() for f in missing]}")

X_train_sel = X_train[top_features]
X_val_sel   = X_val[top_features]

skf       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f2_scorer = make_scorer(fbeta_score, beta=2)

# ──────────────────────────────────────────
# 3. Optuna (원본 데이터로 CV — 뻥튀기 없음)
# ──────────────────────────────────────────
def lgbm_objective(trial):
    params = {
        'device': 'gpu', 'random_state': 42, 'verbose': -1, 'n_jobs': 1,
        'scale_pos_weight': scale_pos,
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1500),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    return cross_val_score(lgb.LGBMClassifier(**params),
                           X_train_sel, y_train,
                           cv=skf, scoring=f2_scorer, n_jobs=1).mean()

def xgb_objective(trial):
    params = {
        'tree_method': 'hist', 'device': 'cuda',
        'random_state': 42, 'verbosity': 0, 'n_jobs': 1,
        'scale_pos_weight': scale_pos,
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 300, 1500),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma':            trial.suggest_float('gamma', 1e-4, 5.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 30),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    return cross_val_score(xgb.XGBClassifier(**params),
                           X_train_sel, y_train,
                           cv=skf, scoring=f2_scorer, n_jobs=1).mean()

pruner = optuna.pruners.MedianPruner(n_startup_trials=5)

print("\nLightGBM Optuna (100 trial)...")
lgbm_study = optuna.create_study(direction='maximize', pruner=pruner)
lgbm_study.optimize(lgbm_objective, n_trials=30, show_progress_bar=True)
print(f"LightGBM CV F2: {lgbm_study.best_value:.4f}")

print("\nXGBoost Optuna (100 trial)...")
xgb_study = optuna.create_study(direction='maximize', pruner=pruner)
xgb_study.optimize(xgb_objective, n_trials=30, show_progress_bar=True)
print(f"XGBoost CV F2: {xgb_study.best_value:.4f}")

# ──────────────────────────────────────────
# 4. 최종 학습 (원본 데이터 — CV와 동일)
# ──────────────────────────────────────────
print("\n최종 모델 학습 중...")

best_lgbm = lgb.LGBMClassifier(
    **lgbm_study.best_params,
    scale_pos_weight=scale_pos,
    device='gpu', random_state=42, verbose=-1, n_jobs=1
)
best_lgbm.fit(X_train_sel, y_train)

best_xgb = xgb.XGBClassifier(
    **xgb_study.best_params,
    scale_pos_weight=scale_pos,
    tree_method='hist', device='cuda',
    random_state=42, verbosity=0, n_jobs=1
)
best_xgb.fit(X_train_sel, y_train)

best_cat = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    scale_pos_weight=scale_pos,
    task_type='GPU', random_seed=42, verbose=0
)
best_cat.fit(X_train_sel, y_train,
             eval_set=(X_val_sel, y_val),
             early_stopping_rounds=50)

# ──────────────────────────────────────────
# 5. 검증 & 임계값 최적화
# ──────────────────────────────────────────
def best_threshold_f2(y_true, y_proba):
    thresholds = np.arange(0.01, 0.80, 0.01)
    scores = [fbeta_score(y_true, (y_proba >= t).astype(int), beta=2)
              for t in thresholds]
    return thresholds[np.argmax(scores)], max(scores)

lgbm_proba = best_lgbm.predict_proba(X_val_sel)[:, 1]
xgb_proba  = best_xgb.predict_proba(X_val_sel)[:, 1]
cat_proba  = best_cat.predict_proba(X_val_sel)[:, 1]

lgbm_thresh, lgbm_f2 = best_threshold_f2(y_val, lgbm_proba)
xgb_thresh,  xgb_f2  = best_threshold_f2(y_val, xgb_proba)
cat_thresh,  cat_f2   = best_threshold_f2(y_val, cat_proba)

ens_proba_val      = (lgbm_proba*0.30) + (xgb_proba*0.50) + (cat_proba*0.20)
ens_thresh, ens_f2 = best_threshold_f2(y_val, ens_proba_val)

print(f"\n{'모델':<12} {'CV F2':>8} {'Val F2':>8} {'AUC':>8} {'Threshold':>10}")
print("-" * 52)
print(f"{'LightGBM':<12} {lgbm_study.best_value:>8.4f} {lgbm_f2:>8.4f} {roc_auc_score(y_val,lgbm_proba):>8.4f} {lgbm_thresh:>10.2f}")
print(f"{'XGBoost':<12} {xgb_study.best_value:>8.4f} {xgb_f2:>8.4f} {roc_auc_score(y_val,xgb_proba):>8.4f} {xgb_thresh:>10.2f}")
print(f"{'CatBoost':<12} {'N/A':>8} {cat_f2:>8.4f} {roc_auc_score(y_val,cat_proba):>8.4f} {cat_thresh:>10.2f}")
print(f"{'앙상블':<12} {'N/A':>8} {ens_f2:>8.4f} {'':>8} {ens_thresh:>10.2f}")

# ──────────────────────────────────────────
# 6. Test 예측 & CSV 저장
# ──────────────────────────────────────────
print("\nTest 예측 중...")
test_df  = pd.read_parquet(TEST_PATH)
test_ids = test_df['ID'].values

X_test_reduced = test_df.drop(columns=to_drop, errors='ignore')
X_test_scaled  = pd.DataFrame(
    scaler.transform(X_test_reduced),
    columns=X_test_reduced.columns
)
X_test_sel = X_test_scaled.drop(columns=['ID'], errors='ignore')[top_features]

lgbm_proba_test = best_lgbm.predict_proba(X_test_sel)[:, 1]
xgb_proba_test  = best_xgb.predict_proba(X_test_sel)[:, 1]
cat_proba_test  = best_cat.predict_proba(X_test_sel)[:, 1]
ens_proba_test  = (lgbm_proba_test*0.30) + (xgb_proba_test*0.50) + (cat_proba_test*0.20)

lgbm_pred = (lgbm_proba_test >= lgbm_thresh).astype(int)
xgb_pred  = (xgb_proba_test  >= xgb_thresh).astype(int)
cat_pred  = (cat_proba_test  >= cat_thresh).astype(int)
ens_pred  = (ens_proba_test  >= ens_thresh).astype(int)

print(f"[LightGBM] 파산: {lgbm_pred.sum()}건")
print(f"[XGBoost]  파산: {xgb_pred.sum()}건")
print(f"[CatBoost] 파산: {cat_pred.sum()}건")
print(f"[앙상블]   파산: {ens_pred.sum()}건")

pd.DataFrame({'ID': test_ids, 'Bankrupt?': lgbm_pred}).to_csv('result_lgbm.csv',    index=False)
pd.DataFrame({'ID': test_ids, 'Bankrupt?': xgb_pred}).to_csv('result_xgb.csv',     index=False)
pd.DataFrame({'ID': test_ids, 'Bankrupt?': cat_pred}).to_csv('result_catboost.csv', index=False)
pd.DataFrame({'ID': test_ids, 'Bankrupt?': ens_pred}).to_csv('result_ensemble.csv', index=False)
print("\n4개 CSV 저장 완료!")
print("→ CV F2랑 Val F2 차이 0.1 이내 + 높은 거 제출!")